# Experiment 2: more varied training data (SeyhaLite + ALT)

Experiment 1 showed near-perfect scores on the in-domain test set but poor translations of everyday sentences unlike the training templates. Experiment 2 adds **ALT** (Asian Language Treebank: 18k professionally translated news sentences in standard Khmer) to the training data and retrains **all three approaches with the same hyperparameters and step budgets** as experiment 1. Only the data changes.

Every model is scored on **two test sets**: the original in-domain test set (same 5,000 sentences as experiment 1) and ALT's out-of-domain news test set (1,018 sentences).

This notebook also scores experiment 1's model C on the ALT test set; the other experiment-1 models were scored locally.

**Run:** Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**. If the session disconnects, reconnect and **Run all** again: finished steps are skipped and training resumes from the latest checkpoint in Drive.

## 0. Settings

In [ ]:
# Hyperparameters tuned in experiment 1, reused unchanged so that only the data differs.
SCRATCH_ARGS = "--learning-rate 0.001 --dropout 0.1"                   # A: 15 epochs, early stopping
FROZEN_ARGS = "--max-steps 16000 --eval-steps 2000 --save-steps 2000"  # B: lr 1e-4 (default)
FULL_ARGS = "--learning-rate 0.0001 --dropout 0.1"                     # C: 2,000 steps (default)

# Delete an approach's checkpoints once its final model is saved, to stay within the Drive quota.
# (Deleted files go to the Drive trash; empty it if Drive runs out of space.)
CLEAN_CHECKPOINTS = True

## 1. GPU, Drive and code

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%cd /content/drive/MyDrive/english-khmer-translation

In [ ]:
import hashlib
import os

# Check that the code in Drive is exactly the version this notebook was built for.
EXPECTED_SHA256 = {
    "requirements.txt": "95d2623eca5fe2b1701ba5bc9c834ad025e1b7c3186fb4a34e04b8d440f65f45",
    "src/__init__.py": "97dccf83164d397e8db471de62e6f0f64e6ac4d77d832ec7cc237184ce7b3421",
    "src/compare.py": "08f591182e64195f82136d9479e21d13a29ed632f0f03f790d34a07dc6c51ee3",
    "src/compare_experiments.py": "9282abe13199b1f2f3de756a4a2d09d71568db4057afabe23a94c8b946eddc7a",
    "src/config.py": "b0cb45232c665aeae68f96e94aa9f686e61b10852415d2d184afe8292f806164",
    "src/data.py": "513fa4a65d42bab0f42588d225a65cefb8728d752d98fcffbd6f868c73e27be8",
    "src/data_stats.py": "f8b313497a7740762f3618509136ee31a8327bcb10d7ae64a1318705d100e333",
    "src/error_analysis.py": "7137f692ed146d7ff522f1c66dbfee7f8448962671e557b0ee19b5041c033616",
    "src/evaluate.py": "f773123651c7c42eb7c4cbf802b4ff4b3dea71af2b2dba734da1d8abf097f3bc",
    "src/model_scratch.py": "bd2408cca0311e37117c1a9a379c4ba56b82b2706e9a92ce679a5f76336bec7d",
    "src/plots.py": "ec35f107c8958d8eb3c5a5df3b75f8d38c1f7f773bdd698409d0dd08100f3a9b",
    "src/probe.py": "5cab2934d2618d353c8979be8ee53c296ce06f033972e3bb0afd08d926358c6e",
    "src/train_finetune.py": "411c2df87a863146cb9e388248e0f5ded19ac2449e937cd5d59e12f32184d51b",
    "src/train_frozen.py": "1b59505a6918da3ebbe96b9bcabef349c0f0441903ac99ca3dac6d390b9a08e2",
    "src/train_scratch.py": "1e090e7a117ee84a941fe3483e3e2c6d4ced36253b6db279ef66cbcccd378c44",
    "src/translate.py": "41fb86eca1590179fe6550119d1f2984e9244f2b0d7dbb7dc5df77118ba70a38",
    "src/tune.py": "faf5bf1e819cc21875bb6fa4592b4be25a40cbf674ddc9d1369fd2ba434ec41a",
    "src/utils/__init__.py": "2260db3bdcd3b4ebd0ffa725c1769cf83a7dfcf33b969d0119002d366ed67dca",
    "src/utils/metrics.py": "782a90e9f4991f6c7ce955e112a28c1b8592dfb3d0a0b27c9ea91ed204259682",
    "src/utils/plotting.py": "d6ffff0bb85b7056d8ac5f10f36d0ac54559a2c3d1d1ad803b983523db5634a8",
    "src/utils/reporting.py": "55e3bb437798e65b2c012ea2e2d31b74a32eebb82739061902e4a30ea57eb750",
    "src/utils/seed.py": "c8f3dd3d3909e9116b415e6214885603ed9fd9cf20b3ab7aebfa6d8b39bacef1",
    "src/utils/training.py": "5249ee051e69a2dc3c46d156b09d7d77c69a15aea5b3a673bdc7526d6e23bacd",
}
problems = [
    f"missing: {p}" if not os.path.isfile(p) else f"changed: {p}"
    for p, h in EXPECTED_SHA256.items()
    if not os.path.isfile(p) or hashlib.sha256(open(p, "rb").read()).hexdigest() != h
]
if problems:
    raise RuntimeError("Code check failed:\n" + "\n".join(problems))
print(f"All {len(EXPECTED_SHA256)} code files verified.")

In [ ]:
!grep -v "^torch==" requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

In [ ]:
import glob
import shutil

# Every `!python` command below inherits this: all outputs go to results/augmented,
# models/augmented and checkpoints/augmented, so experiment 1's files are untouched.
os.environ["EXPERIMENT"] = "augmented"

def done(path: str) -> bool:
    """True if a step's output exists (the step is then skipped)."""
    if os.path.exists(path):
        print(f"✓ {path} exists; skipping.")
        return True
    return False

def resume_flag(pattern: str) -> str:
    """'--resume' if a checkpoint matching pattern exists in Drive."""
    return "--resume" if glob.glob(pattern) else ""

def clean(checkpoint_dir: str, summary_path: str) -> None:
    """Delete an approach's checkpoints, but only once training has finished.

    A failing `!python` line does not stop the cell, so the training summary (written
    only at the very end of a successful run) is checked first: after a crash the
    checkpoints are kept so the next run can resume.
    """
    if CLEAN_CHECKPOINTS and os.path.isfile(summary_path) and os.path.isdir(checkpoint_dir):
        shutil.rmtree(checkpoint_dir)
        print(f"Deleted {checkpoint_dir}")

## 2. Data: SeyhaLite + ALT

In [ ]:
if not done("data/processed/nllb_eng_khm_augmented"):
    !python -m src.data

## 3. Experiment 1's model C on the ALT test set

In [ ]:
if not done("results/full_alt_results.json"):
    !EXPERIMENT=base python -m src.evaluate --approach full --test-set alt

## 4. Approach A: Transformer from scratch

In [ ]:
if not done("results/augmented/scratch_train_summary.json"):
    args = f"{SCRATCH_ARGS} {resume_flag('checkpoints/augmented/scratch/last.pt')}"
    !python -m src.train_scratch {args}
clean("checkpoints/augmented/scratch", "results/augmented/scratch_train_summary.json")
if not done("results/augmented/scratch_results.json"):
    !python -m src.evaluate --approach scratch
if not done("results/augmented/scratch_alt_results.json"):
    !python -m src.evaluate --approach scratch --test-set alt

## 5. Approach B: NLLB frozen backbone

In [ ]:
if not done("results/augmented/frozen_train_summary.json"):
    args = f"{FROZEN_ARGS} {resume_flag('checkpoints/augmented/frozen/checkpoint-*')}"
    !python -m src.train_frozen {args}
clean("checkpoints/augmented/frozen", "results/augmented/frozen_train_summary.json")
if not done("results/augmented/frozen_results.json"):
    !python -m src.evaluate --approach frozen
if not done("results/augmented/frozen_alt_results.json"):
    !python -m src.evaluate --approach frozen --test-set alt

## 6. Approach C: NLLB full fine-tuning

In [ ]:
if not done("results/augmented/full_train_summary.json"):
    args = f"{FULL_ARGS} {resume_flag('checkpoints/augmented/full_finetune/checkpoint-*')}"
    !python -m src.train_finetune {args}
clean("checkpoints/augmented/full_finetune", "results/augmented/full_train_summary.json")
if not done("results/augmented/full_results.json"):
    !python -m src.evaluate --approach full
if not done("results/augmented/full_alt_results.json"):
    !python -m src.evaluate --approach full --test-set alt

## 7. Results bundle
Creates `results_bundle_exp2.zip` in Drive with experiment 2's results, experiment 1's C scores on ALT, and experiment 2's small scratch model. The comparison tables and figures are built locally from this bundle.

In [ ]:
for key in ("scratch", "frozen", "full"):
    for suffix in ("", "_alt"):
        path = f"results/augmented/{key}{suffix}_results.json"
        print(("✓ " if os.path.isfile(path) else "✗ missing ") + path)

!rm -f results_bundle_exp2.zip
!zip -q -r results_bundle_exp2.zip results/augmented results/full_alt_results.json results/predictions/full_alt.jsonl models/augmented/scratch -x "models/augmented/scratch/spm.vocab"
!ls -lh results_bundle_exp2.zip